## Regex-Based Classification

In [1]:
import polars as pl

def classify_cancer_samples(df: pl.DataFrame) -> pl.DataFrame:
    """
    Classify samples as cancer / non-cancer / uncertain based on metadata text patterns.
    Returns the same DataFrame with added classification columns.
    """

    # --- Step 1: Setup --- look
    PRIORITY_COLS = ["source_name", "tissue", "phenotype", "disease", "cell_type", "tumor_type"]
    PRIORITY_COLS = [c for c in PRIORITY_COLS if c in df.columns]

    # Regex patterns
    CANCER_POS = r"(?:\bcancers?\b|\btumou?rs?\b|\bmalignan(?:t|cy)\b|\bcarcinomas?\b|\bneoplasms?\b|\bmetasta(?:s|t)es?\b|\badenocarcinomas?\b|\bsarcomas?\b|\bleukemi(?:a|as)\b|\blymphom(?:a|as)\b|\bglioblastomas?\b|\bmelanomas?\b|\boncolog(?:y|ic|ical)\b)"
    CANCER_NEG = r"(?:\bnormal\b|\bhealthy\b|\bctrl\b|\badjacent normal\b|\bnon[-\s]?tumou?r(?:al)?\b|\bbenign\b|\bnon[-\s]?cancer(?:ous)?\b|\bsham\b|\bunaffected\b)"
    ONCO_TRAPS = r"(?:\boncophora\b|\boncorhynchus\b|\boncotic\b|\boncomodulin\b)"

    # Helper to normalize text columns
    def normalize_text(col_expr):
        return (
            col_expr.cast(pl.Utf8)
            .fill_null("")
            .str.to_lowercase()
            .str.replace_all(r"[_/|]", " ")
            .str.replace_all(r"\s+", " ")
            .str.strip_chars()
        )

    # --- Step 2: Sample name detection ---
    sample_name_col = "title" if "title" in df.columns else "biosample"

    df = df.with_columns([
        normalize_text(pl.col(sample_name_col)).str.contains(CANCER_POS).alias("cancer_in_sample_name"),
        normalize_text(pl.col(sample_name_col)).str.contains(CANCER_NEG).alias("negative_in_sample_name"),
        normalize_text(pl.col(sample_name_col)).str.contains(ONCO_TRAPS).alias("onco_trap_in_sample_name"),
    ])

    # --- Step 3: Check priority columns ---
    for col in PRIORITY_COLS:
        df = df.with_columns([
            normalize_text(pl.col(col)).str.contains(CANCER_POS).alias(f"cancer_in_{col}"),
            normalize_text(pl.col(col)).str.contains(CANCER_NEG).alias(f"negative_in_{col}"),
        ])

    # --- Step 4: Count mentions ---
    cancer_mention_cols = [f"cancer_in_{c}" for c in PRIORITY_COLS]
    negative_mention_cols = [f"negative_in_{c}" for c in PRIORITY_COLS]

    df = df.with_columns([
        pl.sum_horizontal([pl.col(c) for c in cancer_mention_cols if c in df.columns]).alias("n_cancer_mentions"),
        pl.sum_horizontal([pl.col(c) for c in negative_mention_cols if c in df.columns]).alias("n_negative_mentions"),
    ])

    # --- Step 5: Confidence category ---
    df = df.with_columns([
        pl.when(pl.col("onco_trap_in_sample_name"))
        .then(pl.lit("uncertain_onco_trap"))
        .when(
            pl.col("cancer_in_sample_name") &
            (pl.col("n_cancer_mentions") >= 1) &
            (pl.col("n_negative_mentions") == 0)
        )
        .then(pl.lit("confident_cancer"))
        .when(
            (pl.col("cancer_in_sample_name") & (pl.col("n_negative_mentions") == 0)) |
            (pl.col("n_cancer_mentions") >= 2)
        )
        .then(pl.lit("likely_cancer"))
        .when(
            pl.col("negative_in_sample_name") |
            (pl.col("n_negative_mentions") >= 1)
        )
        .then(pl.lit("likely_non_cancer"))
        .when(pl.col("n_cancer_mentions") == 1)
        .then(pl.lit("uncertain_weak_signal"))
        .otherwise(pl.lit("uncertain_no_signal"))
        .alias("confidence_category")
    ])

    return df


## Load Data

In [4]:
full_dataset = pl.read_csv(
    "data/combined_metadata_noncancer_removed.csv",
    schema_overrides={"group": pl.Utf8},   # <- dict of {col_name: dtype}
    infer_schema_length=0,                  # scan entire file for other cols
)

df = pl.read_csv("data/train_test.csv")

In [5]:
df = df[["experiment_accession", "bioproject", "biosample", "sample_accession","run_accession", "is_cancer"]]

# idx = full_dataset.columns.index("cancer_type")

# full_dataset = full_dataset[full_dataset.columns[:idx+1]]

In [6]:
df.head()

# full_dataset.head()

experiment_accession,bioproject,biosample,sample_accession,run_accession,is_cancer
str,str,str,str,str,i64
"""SRX10489019""","""PRJNA718814""","""GSM5220969""","""SRS8615268""","""SRR14118670""",1
"""ERX1283444""","""PRJEB6455""","""SAMEA3724868""","""ERS1032017""","""ERR1211193""",0
"""SRX17708205""","""PRJNA884362""","""GSM6601021""","""SRS15238722""","""SRR21710838""",1
"""SRX4518392""","""PRJNA484951""","""GSM3323504""","""SRS3636903""","""SRR7655999""",0
"""SRX9364181""","""PRJNA671877""","""GSM4860401""","""SRS7586602""","""SRR12899131""",0


In [7]:
joined = full_dataset.join(
    df,
    on=["experiment_accession", "bioproject", "biosample", "sample_accession", "run_accession"],  # matching keys
    how="inner"  # keep only matching rows
)

## Do Prediction

In [8]:
predicted_df = classify_cancer_samples(joined)

In [9]:
# find the position of "cancer_type"
idx = predicted_df.columns.index("cancer_type")

# slice up to that column (inclusive)
cols_to_keep = predicted_df.columns[:idx + 1]

# add the two specific extra columns
cols_to_keep += ["source_name", "is_cancer", "confidence_category"]

predicted_df.select(cols_to_keep)



experiment_alias,experiment_accession,title,study_name,sample_accession,library_name,library_strategy,library_source,library_selection,library_layout,platform_instrument_model,submitter_accession,submitter_id,organization_type,organization_name,study_accession,biosample,bioproject,organism_tax_id,organism_scientific_name,breed,dev_stage,sex,tissue,biosamplemodel,run_accession,run_total_spots,run_total_bases,run_size,run_load_done,run_is_public,run_has_tax_analysis,a_count,c_count,g_count,t_count,n_count,cancer_type,source_name,is_cancer,confidence_category
str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,i64,str
"""ena-EXPERIMENT-FUNCTIONAL GENO…","""ERX3526057""","""Illumina HiSeq 2500 paired end…",null,"""ERS3719163""","""2_Normal""","""RNA-Seq""","""TRANSCRIPTOMIC""","""other""","""PAIRED""","""Illumina HiSeq 2500""","""ERA2111341""","""2_Normal ena-SUBMISSION-FUNCTI…","""center""","""FUNCTIONAL GENOMICS CENTER ZUR…","""ERP117109""","""SAMEA5930682""","""PRJEB34234""","""9615""","""Canis lupus familiaris""",null,null,"""nan nan nan nan nan nan""",null,null,"""ERR3504843""","""13198053""","""3985812006""","""2403610007""","""TRUE""","""TRUE""","""1""","""968718068""","""1023813547""","""1112291056""","""878463736""","""2525599""","""carcinoma""",null,0,"""likely_non_cancer"""
"""GSM4860400""","""SRX9364180""","""GSM4860400: Fresh-frozen canin…",null,"""SRS7586601""",null,"""RNA-Seq""","""TRANSCRIPTOMIC""","""cDNA""","""PAIRED""","""Illumina NovaSeq 6000""","""SRA1147864""","""nan GEO: GSE160124 nan nan GEO…","""center""","""NCBI""","""SRP288597""","""GSM4860400""","""PRJNA671877""","""9615""","""Canis lupus familiaris""","""Hovawart""",null,"""nan nan nan nan nan nan""","""nan control nan nan control na…",null,"""SRR12899130""","""32802407""","""3292101545""","""1101440073""","""TRUE""","""TRUE""","""1""","""798070483""","""761998966""","""845415170""","""879829853""","""6787073""","""tumor*""","""Fresh-frozen canine healthy br…",0,"""likely_non_cancer"""
"""GSM4860401""","""SRX9364181""","""GSM4860401: Fresh-frozen canin…",null,"""SRS7586602""",null,"""RNA-Seq""","""TRANSCRIPTOMIC""","""cDNA""","""PAIRED""","""Illumina NovaSeq 6000""","""SRA1147864""","""nan GEO: GSE160124 nan nan GEO…","""center""","""NCBI""","""SRP288597""","""GSM4860401""","""PRJNA671877""","""9615""","""Canis lupus familiaris""","""French Bulldog""",null,"""nan nan nan nan nan nan""","""nan control nan nan control na…",null,"""SRR12899131""","""36359099""","""3612160297""","""1220312604""","""TRUE""","""TRUE""","""1""","""889453430""","""826675433""","""915664165""","""965546260""","""14821009""","""tumor*""","""Fresh-frozen canine healthy br…",0,"""likely_non_cancer"""
"""GSM3005111""","""SRX13999782""","""GSM3005111: D_Normal_Mucosa; C…",null,"""SRS11838164""",null,"""RNA-Seq""","""TRANSCRIPTOMIC""","""cDNA""","""PAIRED""","""Illumina HiSeq 4000""","""SRA1364912""","""GEO: GSE110661""","""center""","""NCBI""","""SRP357615""","""SAMN08550188""","""PRJNA434265""","""9615""","""Canis lupus familiaris""",null,null,"""nan nan nan nan nan nan""","""Bladder""",null,"""SRR17838664""","""9096025""","""2461885361""","""1015954149""","""TRUE""","""TRUE""",null,"""695834791""","""506960216""","""511045330""","""704969084""","""2648204""","""bladder cancer""","""Normal Urothelial cells""",0,"""likely_non_cancer"""
"""GSM2498357""","""SRX2581899""","""GSM2498357: CHAD-P9; Canis lup…",null,"""SRS1995924""",null,"""RNA-Seq""","""TRANSCRIPTOMIC""","""cDNA""","""PAIRED""","""Illumina HiSeq 2000""","""SRA539403""","""nan GEO: GSE95183 nan nan GEO:…","""center""","""NCBI""","""SRP100514""","""GSM2498357""","""PRJNA376380""","""9615""","""Canis lupus familiaris""","""Portuguese Water Dog""",null,"""nan nan nan nan nan nan""","""nan spleen nan nan spleen nan""",null,"""SRR5278015""","""22714044""","""2271404400""","""1615616886""","""TRUE""","""TRUE""","""1""","""5787

In [10]:
    # Create sample_id if it doesn't exist (use sample_accession as unique identifier)
if "sample_id" not in predicted_df.columns:
    predicted_df = predicted_df.with_columns(
        pl.col("sample_accession").alias("sample_id")
    )

uncertain_df = predicted_df.filter(
    pl.col("confidence_category").is_in(["uncertain_no_signal", "uncertain_weak_signal", "likely_non_cancer"])
)

## Define Cancer Rules

In [11]:
import medspacy
from medspacy.ner import TargetRule
from medspacy.target_matcher import TargetMatcher

nlp = medspacy.load(enable=["ner", "context"])
tm = nlp.get_pipe("medspacy_target_matcher")

rules = [
    # General cancer terms
    TargetRule(
        literal="cancer",
        category="CANCER",
        pattern=[{"LOWER": "cancer"}]
    ),
    TargetRule(
        literal="tumor",
        category="CANCER",
        pattern=[{"LOWER": {"IN": ["tumor", "tumour", "tumors", "tumours"]}}]
    ),
    TargetRule(
        literal="carcinoma",
        category="CANCER",
        pattern=[{"LOWER": {"REGEX": "carcinomas?"}}]
    ),
    
    # Specific cancer types
    TargetRule(
        literal="adenocarcinoma",
        category="CANCER",
        pattern=[{"LOWER": {"REGEX": "adenocarcinomas?"}}]
    ),
    TargetRule(
        literal="squamous cell carcinoma",
        category="CANCER",
        pattern=[
            {"LOWER": "squamous"},
            {"LOWER": "cell"},
            {"LOWER": {"REGEX": "carcinomas?"}}
        ]
    ),
    TargetRule(
        literal="small cell carcinoma",
        category="CANCER",
        pattern=[
            {"LOWER": "small"},
            {"LOWER": "cell"},
            {"LOWER": {"REGEX": "carcinomas?"}}
        ]
    ),
    TargetRule(
        literal="non-small cell carcinoma",
        category="CANCER",
        pattern=[
            {"LOWER": "non"},
            {"IS_PUNCT": True, "OP": "?"},  # optional hyphen
            {"LOWER": "small"},
            {"LOWER": "cell"},
            {"LOWER": {"REGEX": "carcinomas?"}}
        ]
    ),
    
    # Leukemia/Lymphoma
    TargetRule(
        literal="leukemia",
        category="CANCER",
        pattern=[{"LOWER": {"REGEX": "leuk[ae]mias?"}}]
    ),
    TargetRule(
        literal="lymphoma",
        category="CANCER",
        pattern=[{"LOWER": {"REGEX": "lymphomas?"}}]
    ),
    TargetRule(
        literal="acute myeloid leukemia",
        category="CANCER",
        pattern=[
            {"LOWER": "acute"},
            {"LOWER": "myeloid"},
            {"LOWER": {"REGEX": "leuk[ae]mias?"}}
        ]
    ),
    
    # Sarcomas
    TargetRule(
        literal="sarcoma",
        category="CANCER",
        pattern=[{"LOWER": {"REGEX": "sarcomas?"}}]
    ),
    TargetRule(
        literal="osteosarcoma",
        category="CANCER",
        pattern=[{"LOWER": {"REGEX": "osteosarcomas?"}}]
    ),
    
    # Brain tumors
    TargetRule(
        literal="glioblastoma",
        category="CANCER",
        pattern=[{"LOWER": {"REGEX": "glioblastomas?"}}]
    ),
    TargetRule(
        literal="glioma",
        category="CANCER",
        pattern=[{"LOWER": {"REGEX": "gliomas?"}}]
    ),
    
    # Melanoma
    TargetRule(
        literal="melanoma",
        category="CANCER",
        pattern=[{"LOWER": {"REGEX": "melanomas?"}}]
    ),
    
    # Malignancy terms
    TargetRule(
        literal="malignant",
        category="CANCER",
        pattern=[{"LOWER": {"REGEX": r"(?<!pre[- ])malignan(t|cy)"}}]
    ),

    TargetRule(
        literal="neoplasm",
        category="CANCER",
        pattern=[{"LOWER": {"REGEX": "neoplasms?"}}]
    ),
    TargetRule(
        literal="metastasis",
        category="CANCER",
        pattern=[{"LOWER": {"REGEX": "metasta(sis|ses|tic)"}}]
    ),
    
    # Context-dependent patterns (adjective + tissue)
    TargetRule(
        literal="malignant tissue",
        category="CANCER",
        pattern=[
            {"LOWER": "malignant"},
            {"LOWER": {"IN": ["tissue", "cells", "lesion", "mass"]}}
        ]
    ),
    TargetRule(
        literal="cancerous tissue",
        category="CANCER",
        pattern=[
            {"LOWER": "cancerous"},
            {"LOWER": {"IN": ["tissue", "cells", "lesion", "mass"]}}
        ]
    ),
    
    # Oncology context
    TargetRule(
        literal="oncology",
        category="CANCER",
        pattern=[{"LOWER": {"REGEX": "oncolog(y|ic|ical)"}}],
        # This one might need more context checking
    ),
    
    # Cell line patterns (common in research)
    TargetRule(
        literal="cancer cell line",
        category="CANCER",
        pattern=[
            {"LOWER": {"IN": ["cancer", "tumor", "tumour"]}},
            {"LOWER": "cell"},
            {"LOWER": {"IN": ["line", "lines"]}}
        ]
    ),

    TargetRule(
        literal="TIL",
        category="CANCER",
        pattern=[
            {"LOWER": {"IN": ["til", "tils", "t-i-l", "t.i.l.", "t.i.l.s."]}},
            {"LOWER": {"IN": ["tumor", "tumour"]}, "OP": "?"},
            {"LOWER": {"IN": ["infiltrating", "infiltrated"]}, "OP": "?"},
            {"LOWER": "lymphocytes", "OP": "?"},
        ]
    )

]

tm.add(rules)

## Define non-cancer rules

In [12]:
non_cancer_rules = [
    TargetRule(
        literal="normal tissue",
        category="NON_CANCER",
        pattern=[
            {"LOWER": {"IN": ["normal", "healthy", "control", "benign", "adjacent"]}},
            {"LOWER": {"IN": ["tissue", "sample", "cells", "fat", "pad", "organ"]}, "OP": "?"}
        ]
    ),

    TargetRule(
        literal="benign lesion",
        category="NON_CANCER",
        pattern=[
            {"LOWER": "benign"},
            {"LOWER": {"IN": ["lesion", "mass", "tumor", "tumour"]}}
        ]
    ),

    TargetRule(
    literal="premalignant",
    category="NON_CANCER",
    pattern=[{"LOWER": {"REGEX": "pre[- ]?malignan(t|cy)"}}]
    )

]

tm.add(non_cancer_rules)

## MedSpacy Helper Funcs

In [13]:
def medspacy_classify(row_texts):
    cancer_found = False
    non_cancer_found = False
    negation_found = False

    for text in row_texts:
        doc = nlp(text)
        for ent in doc.ents:
            if ent.label_ == "CANCER":
                if ent._.is_negated:
                    negation_found = True
                else:
                    cancer_found = True
            elif ent.label_ == "NON_CANCER":
                if not ent._.is_negated:
                    non_cancer_found = True

    # Decision hierarchy
    if cancer_found and not negation_found:
        return "CANCER"
    elif non_cancer_found and not cancer_found:
        return "NOT_CANCER"
    elif cancer_found and non_cancer_found:
        return "UNCERTAIN"
    elif negation_found:
        return "NOT_CANCER"
    else:
        return "NO_SIGNAL"


UNCERTAIN_LABELS = [
    "uncertain_no_signal",
    "uncertain_weak_signal"
]

def clean_texts(row):
    texts = [
        str(row[col]).strip()
        for col in ["source_name", "tissue", "phenotype", "disease"]
        if col in row and row[col] not in (None, "None", "nan", "NaN", "", "null")
    ]
    return texts


def resolve_uncertain(regex_label: str, med_label: str | None) -> str:
    if med_label is None:
        return regex_label

    if regex_label in UNCERTAIN_LABELS:
        if med_label == "CANCER":
            return "confirmed_by_medspacy"
        elif med_label == "NOT_CANCER":
            return "confirmed_non_cancer"
        elif med_label == "UNCERTAIN":
            return "uncertain_medspacy"
        else:
            return regex_label

    # Confident non-cancer but medspacy says otherwise — flip if strong contradiction
    if regex_label == "likely_non_cancer" and med_label == "CANCER":
        return "confirmed_by_medspacy"

    return regex_label

## Run the MedSpacy on each of my rows of interests

In [14]:
# uncertain_df = uncertain_df.filter(pl.col("source_name") == "Mus musculus CD8 TILs")

In [15]:
results = []

for row in uncertain_df.iter_rows(named=True):
    texts = clean_texts(row)
    if texts:  # skip empty rows
        result = medspacy_classify(texts)
    else:
        result = "NO_SIGNAL"
    results.append(result)

2025-11-16 13:54:38.613 | DEBUG    | PyRuSH.PyRuSHSentencizer:predict:100 - [cpredict_split_gaps|call_id=0] [doc 0] Token 0 'Fresh' marked as sentence start (span begin)
2025-11-16 13:54:38.613 | DEBUG    | PyRuSH.PyRuSHSentencizer:predict:100 - [cpredict_split_gaps|call_id=0] Token/tag mapping: [(Fresh, True), (-, False), (frozen, False), (canine, False), (healthy, False), (brain, False), (tissue, False)]
2025-11-16 13:54:38.614 | DEBUG    | PyRuSH.PyRuSHSentencizer:predict:100 - [cpredict_split_gaps|call_id=1] [doc 0] Token 0 'nan' marked as sentence start (span begin)
2025-11-16 13:54:38.615 | DEBUG    | PyRuSH.PyRuSHSentencizer:predict:100 - [cpredict_split_gaps|call_id=1] Token/tag mapping: [(nan, True), (control, False), (nan, False), (nan, False), (control, False), (nan, False)]
2025-11-16 13:54:38.616 | DEBUG    | PyRuSH.PyRuSHSentencizer:predict:100 - [cpredict_split_gaps|call_id=2] [doc 0] Token 0 'Fresh' marked as sentence start (span begin)
2025-11-16 13:54:38.616 | DEBUG  

In [16]:
uncertain_df = uncertain_df.with_columns(pl.Series("medspacy_detected_cancer", results))

In [17]:
uncertain_df.select(['source_name', 'medspacy_detected_cancer'])

source_name,medspacy_detected_cancer
str,str
null,"""NO_SIGNAL"""
"""Fresh-frozen canine healthy br…","""NOT_CANCER"""
"""Fresh-frozen canine healthy br…","""NOT_CANCER"""
"""Normal Urothelial cells""","""NOT_CANCER"""
"""hemangiosarcoma""","""CANCER"""
…,…
"""WT mammary gland""","""NO_SIGNAL"""
"""Rosa26LSL-Sox2-IRES-GFP;Lkb1fl…","""NO_SIGNAL"""
"""tumor""","""CANCER"""


In [18]:
uncertain_df = uncertain_df.unique(subset=["run_accession"], keep="first")

predicted_df = predicted_df.join(
    uncertain_df.select(["run_accession", "medspacy_detected_cancer"]),
    on=["run_accession"],
    how="left",
)

In [19]:
predicted_df.select(['source_name', 'medspacy_detected_cancer'])

source_name,medspacy_detected_cancer
str,str
null,"""NO_SIGNAL"""
"""Fresh-frozen canine healthy br…","""NOT_CANCER"""
"""Fresh-frozen canine healthy br…","""NOT_CANCER"""
"""Normal Urothelial cells""","""NOT_CANCER"""
"""hemangiosarcoma""","""CANCER"""
…,…
"""tumor""","""CANCER"""
"""cells sorted from MC38 tumors""","""CANCER"""
"""cells sorted from MC38 tumors""","""CANCER"""


In [20]:
predicted_df = predicted_df.with_columns(
    pl.struct(["confidence_category", "medspacy_detected_cancer"])
    .map_elements(
        lambda row: resolve_uncertain(
            row["confidence_category"], row["medspacy_detected_cancer"]
        ),
        return_dtype=pl.Utf8,
    )
    .alias("final_classification")
)

In [21]:
# find the position of "cancer_type"
idx = predicted_df.columns.index("cancer_type")

# slice up to that column (inclusive)
# cols_to_keep = predicted_df.columns[:idx + 1]
cols_to_keep = ["experiment_alias", "source_name", "tissue", "phenotype", "disease"]


# add the two specific extra columns
cols_to_keep += ["is_cancer", "final_classification", "medspacy_detected_cancer"]

In [22]:
predicted_df.select(cols_to_keep)

experiment_alias,source_name,tissue,phenotype,disease,is_cancer,final_classification,medspacy_detected_cancer
str,str,str,str,str,i64,str,str
"""ena-EXPERIMENT-FUNCTIONAL GENO…",null,null,null,null,0,"""likely_non_cancer""","""NO_SIGNAL"""
"""GSM4860400""","""Fresh-frozen canine healthy br…","""nan control nan nan control na…",null,null,0,"""likely_non_cancer""","""NOT_CANCER"""
"""GSM4860401""","""Fresh-frozen canine healthy br…","""nan control nan nan control na…",null,null,0,"""likely_non_cancer""","""NOT_CANCER"""
"""GSM3005111""","""Normal Urothelial cells""","""Bladder""",null,null,0,"""likely_non_cancer""","""NOT_CANCER"""
"""GSM2498357""","""hemangiosarcoma""","""nan spleen nan nan spleen nan""",null,null,1,"""confirmed_by_medspacy""","""CANCER"""
…,…,…,…,…,…,…,…
"""GSM3187876""","""tumor""","""alveolar rhabdomyosarcoma""",null,null,1,"""confirmed_by_medspacy""","""CANCER"""
"""GSM3573140""","""cells sorted from MC38 tumors""","""nan nan nan nan""",null,null,1,"""confirmed_by_medspacy""","""CANCER"""
"""GSM3573150""","""cells sorted from MC38 tumors""","""nan nan nan nan""",null,null,1,"""confirmed_by_medspacy""","""CANCER"""


In [23]:
predicted_df_filtered = (
    predicted_df
    .filter(pl.col("final_classification") == "confirmed_by_medspacy")
    .select(cols_to_keep)
)

predicted_df_filtered

experiment_alias,source_name,tissue,phenotype,disease,is_cancer,final_classification,medspacy_detected_cancer
str,str,str,str,str,i64,str,str
"""GSM2498357""","""hemangiosarcoma""","""nan spleen nan nan spleen nan""",null,null,1,"""confirmed_by_medspacy""","""CANCER"""
"""GSM3993105""","""canine simple mammary carcinom…",null,null,null,1,"""confirmed_by_medspacy""","""CANCER"""
"""GSM5957075_r1""","""Primary murine osteosarcomas""","""Primary tumor""",null,null,1,"""confirmed_by_medspacy""","""CANCER"""
"""E-MTAB-7279:Sample 11_s""",null,"""nan nan nan nan""","""human tumor necrosis factor ex…","""rheumatoid arthritis""",0,"""confirmed_by_medspacy""","""CANCER"""
"""E-MTAB-9353:17038-0023_s""",null,null,"""wild type""","""glioblastoma multiforme""",1,"""confirmed_by_medspacy""","""CANCER"""
…,…,…,…,…,…,…,…
"""GSM3119932""","""Tumor B16F10 melanoma""",null,null,null,1,"""confirmed_by_medspacy""","""CANCER"""
"""GSM3142052""","""mice brain tumors""",null,null,null,1,"""confirmed_by_medspacy""","""CANCER"""
"""GSM3187876""","""tumor""","""alveolar rhabdomyosarcoma""",null,null,1,"""confirmed_by_medspacy""","""CANCER"""


## Looking @ Accuracy

In [ ]:
# Validation: Compare predictions against ground truth
validation_summary = (
    predicted_df
    .group_by("final_classification", "is_cancer")
    .agg(pl.count().alias("count"))
    .sort("final_classification", "is_cancer")
)

print("=== Classification vs Ground Truth ===")
print(validation_summary)

# Calculate accuracy for confident predictions
confident_predictions = predicted_df.filter(
    pl.col("final_classification").is_in(["confident_cancer", "confirmed_by_medspacy"])
)

if len(confident_predictions) > 0:
    correct = confident_predictions.filter(pl.col("is_cancer") == True).height
    total = confident_predictions.height
    accuracy = correct / total * 100
    print(f"\nAccuracy on confident cancer predictions: {accuracy:.1f}% ({correct}/{total})")

# Check non-cancer predictions
non_cancer_predictions = predicted_df.filter(
    pl.col("final_classification").is_in(["likely_non_cancer", "confirmed_non_cancer"])
)

if len(non_cancer_predictions) > 0:
    correct_non = non_cancer_predictions.filter(pl.col("is_cancer") == False).height
    total_non = non_cancer_predictions.height
    accuracy_non = correct_non / total_non * 100
    print(f"Accuracy on non-cancer predictions: {accuracy_non:.1f}% ({correct_non}/{total_non})")

In [ ]:
test_samples = [
    ["breast cancer tissue"],
    ["normal breast tissue"],
    ["no evidence of tumor"],
    ["benign lesion no malignancy"],
    ["metastatic carcinoma"],
    ["healthy control sample"],
    ["previous cancer but resolved"],
    ["adjacent non-tumor tissue"],
    ["Normal mammary fat pad nan Normal mammary fat pad nan"],
    ["null"],
    ["Mus musculus CD8 TILs"],
    ["premalignant stage fallopian tubes"]
]


In [ ]:
for texts in test_samples:
    result = medspacy_classify(texts)
    print(f"{texts[0]:35s} → {result}")

In [ ]:
for row in uncertain_df.iter_rows(named=True):
    texts = clean_texts(row)
    result = medspacy_classify(texts)
    if "TILs" in str(texts):
        print(row["sample_id"], "texts:", texts, "→", result)

## Getting all the Terms from disease row in full dataset

In [ ]:
all_samples = pl.read_csv(
    "data/combined_metadata_noncancer_removed.csv",
    schema_overrides={"group": pl.Utf8},   # <- dict of {col_name: dtype}
    infer_schema_length=0,                  # scan entire file for other cols
)

In [ ]:
unique_diseases = all_samples.select("disease").unique().to_series().to_list()

# TODO: add "cancer_type" - keep the NON_CANCER from disease & base CANCER rules from cancer_type

In [ ]:
len(unique_diseases)

In [ ]:
unique_diseases

## Autogenerate some TargetRules for our list of unique diseases

In [ ]:
## Auto-generate TargetRules for diseases

KEYWORDS_CANCER = (
    "cancer",
    "carcinoma",
    "sarcoma",
    "leuk",
    "lymphoma",
    "tumor",
    "tumour",
    "melanoma",
    "blastoma",
    "myeloma",
    "metast",
)

_skip_literals = {rule.literal.lower() for rule in (rules + non_cancer_rules)}


def _phrase_to_pattern(phrase: str):
    doc = nlp.make_doc(phrase.lower())
    pattern = []
    for token in doc:
        if token.is_space:
            continue
        if token.is_alpha:
            pattern.append({"LOWER": token.lower_})
        elif token.is_digit:
            pattern.append({"LIKE_NUM": True})
        elif token.is_punct:
            pattern.append({"TEXT": token.text})
        else:
            pattern.append({"LOWER": token.lower_})
    return pattern


auto_rules = []
skipped_literals = []

for disease in unique_diseases:
    if disease is None:
        continue
    disease_str = str(disease).strip()
    if not disease_str or disease_str.lower() in {"nan", "none", "null"}:
        continue
    norm_literal = disease_str.lower()
    if norm_literal in _skip_literals:
        skipped_literals.append(disease_str)
        continue
    pattern = _phrase_to_pattern(disease_str)
    if not pattern:
        continue
    category = (
        "CANCER"
        if any(keyword in norm_literal for keyword in KEYWORDS_CANCER)
        else "NON_CANCER"
    )
    auto_rules.append(
        TargetRule(
            literal=disease_str,
            category=category,
            pattern=pattern,
        )
    )

if auto_rules:
    tm.add(auto_rules)

print(f"Added {len(auto_rules)} disease rules, skipped {len(skipped_literals)} existing literals.")

In [ ]:
for r in tm.rules:
    if r.category == "NON_CANCER":
        print(r.literal)


In [ ]:
# Existing TargetRules
# skipped_literals

In [ ]:
predicted_df = classify_cancer_samples(all_samples)

In [ ]:
# find the position of "cancer_type"
idx = predicted_df.columns.index("cancer_type")

# slice up to that column (inclusive)
cols_to_keep = predicted_df.columns[:idx + 1]

# add the two specific extra columns
cols_to_keep += ["source_name", "confidence_category"]

predicted_df.select(cols_to_keep)

# Create sample_id if it doesn't exist (use sample_accession as unique identifier)
if "sample_id" not in predicted_df.columns:
    predicted_df = predicted_df.with_columns(
        pl.col("sample_accession").alias("sample_id")
    )

uncertain_df = predicted_df.filter(
    pl.col("confidence_category").is_in(["uncertain_no_signal", "uncertain_weak_signal", "likely_non_cancer"])
)

results = []

for row in uncertain_df.iter_rows(named=True):
    texts = clean_texts(row)
    if texts:  # skip empty rows
        result = medspacy_classify(texts)
    else:
        result = "NO_SIGNAL"
    results.append(result)

uncertain_df = uncertain_df.with_columns(pl.Series("medspacy_detected_cancer", results))

uncertain_df = uncertain_df.unique(subset=["run_accession"], keep="first")

predicted_df = predicted_df.join(
    uncertain_df.select(["run_accession", "medspacy_detected_cancer"]),
    on=["run_accession"],
    how="left",
)

In [ ]:
predicted_df = predicted_df.with_columns(
    pl.struct(["confidence_category", "medspacy_detected_cancer"])
    .map_elements(
        lambda row: resolve_uncertain(
            row["confidence_category"], row["medspacy_detected_cancer"]
        ),
        return_dtype=pl.Utf8,
    )
    .alias("final_classification")
)

In [ ]:
# find the position of "cancer_type"
idx = predicted_df.columns.index("cancer_type")

# slice up to that column (inclusive)
# cols_to_keep = predicted_df.columns[:idx + 1]
cols_to_keep = ["experiment_alias", "source_name", "tissue", "phenotype", "disease"]


# add the two specific extra columns
cols_to_keep += ["final_classification", "medspacy_detected_cancer"]

predicted_df.select(cols_to_keep)

predicted_df_filtered = (
    predicted_df
    .filter(pl.col("final_classification") == "confirmed_by_medspacy")
    .select(cols_to_keep)
)

predicted_df_filtered